# Streaming Pretraining Metrics Dashboard

Read protein pretraining metric JSONL files and render training charts. This notebook supports the streaming pretrain metrics file written by `03a_streaming_train_step_by_step.ipynb` and the older pretrain/resume metrics files.

## 1. Imports And Paths

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/Microbial DNA Compiler"),
        Path("/content/Microbial-DNA-Compiler"),
        Path("/content/drive/MyDrive/Microbial DNA Compiler"),
        Path("/content/drive/MyDrive/Microbial-DNA-Compiler"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError(
        "Could not locate repo. Run inside the repo, set MDNAC_REPO_DIR, "
        "or in Colab clone/mount the repo under /content or Google Drive."
    )


PROJECT_ROOT = find_repo_dir_for_import(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.notebook_runtime import bootstrap_notebook
from libs.core.pretrain.training_config import load_protein_training_config

RUNTIME = bootstrap_notebook(PROJECT_ROOT)
PROJECT_ROOT = Path(RUNTIME["repo_dir"])
CONFIG_PATH = Path(os.environ.get("MDNAC_TRAIN_CONFIG", PROJECT_ROOT / "train.16gb.yaml")).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (PROJECT_ROOT / CONFIG_PATH).resolve()
TRAINING_CONFIG = load_protein_training_config(PROJECT_ROOT, config_path=CONFIG_PATH)
CHECKPOINT_DIR = TRAINING_CONFIG["paths"]["checkpoint_dir"]
PLOT_DIR = CHECKPOINT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

METRIC_SPECS = [
    ("metrics_history", CHECKPOINT_DIR / "metrics_history.jsonl"),
    ("streaming", CHECKPOINT_DIR / "streaming_training_metrics.jsonl"),
    ("pretrain", CHECKPOINT_DIR / "training_metrics.jsonl"),
    ("resume", CHECKPOINT_DIR / "resume_training_metrics.jsonl"),
]

{
    "runtime": RUNTIME,
    "config_path": str(CONFIG_PATH),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "plot_dir": str(PLOT_DIR),
    "metric_files": {name: str(path) for name, path in METRIC_SPECS},
}


## 2. Load Metric Rows

In [ ]:
def normalize_float(value) -> float:
    if value is None:
        return math.nan
    if isinstance(value, str) and value.strip().lower() in {"", "nan", "none", "null"}:
        return math.nan
    return float(value)


def normalize_int(value, default: int = 0) -> int:
    if value is None or value == "":
        return default
    return int(value)


rows = []
loaded_files = []
for source_index, (source_name, metric_path) in enumerate(METRIC_SPECS):
    if not metric_path.exists():
        continue
    loaded_files.append(str(metric_path))
    for line_number, raw_line in enumerate(metric_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not raw_line.strip():
            continue
        row = json.loads(raw_line)
        row["source"] = source_name
        row["source_index"] = source_index
        row["metric_file"] = str(metric_path)
        row["line_number"] = line_number
        row["epoch"] = normalize_int(row.get("epoch"), default=0)
        row["global_step"] = normalize_int(row.get("global_step"), default=0)
        row["tokens_seen"] = normalize_int(row.get("tokens_seen"), default=0)
        row["train_loss"] = normalize_float(row.get("train_loss"))
        row["val_loss"] = normalize_float(row.get("val_loss"))
        rows.append(row)

if not rows:
    raise FileNotFoundError(
        f"No metric JSONL files found under {CHECKPOINT_DIR}. Run pretraining first."
    )

rows = sorted(rows, key=lambda row: (row["global_step"], row["source_index"], row["line_number"]))
print(f"Loaded {len(rows)} metric rows from {len(loaded_files)} files")
loaded_files

## 3. Training Summary

In [ ]:
def finite(value: float) -> bool:
    return isinstance(value, (int, float)) and math.isfinite(float(value))


def finite_values(key: str) -> list[float]:
    return [float(row[key]) for row in rows if finite(row.get(key, math.nan))]


last_row = rows[-1]
train_losses = finite_values("train_loss")
val_losses = finite_values("val_loss")
summary = {
    "rows": len(rows),
    "sources": sorted({row["source"] for row in rows}),
    "global_step_min": min(row["global_step"] for row in rows),
    "global_step_max": max(row["global_step"] for row in rows),
    "tokens_seen_max": max(row["tokens_seen"] for row in rows),
    "last_epoch": last_row["epoch"],
    "last_global_step": last_row["global_step"],
    "last_tokens_seen": last_row["tokens_seen"],
    "last_train_loss": last_row["train_loss"],
    "best_train_loss": min(train_losses) if train_losses else math.nan,
    "best_val_loss": min(val_losses) if val_losses else math.nan,
}
summary

## 4. Plot Helpers

In [ ]:
SOURCE_ORDER = [source_name for source_name, _ in METRIC_SPECS]
SOURCE_COLORS = {
    "streaming": "tab:blue",
    "pretrain": "tab:green",
    "resume": "tab:orange",
}


def source_rows(source_name: str) -> list[dict[str, object]]:
    return [row for row in rows if row["source"] == source_name]


def xy_series(source_name: str, x_key: str, y_key: str) -> tuple[list[int], list[float]]:
    xs: list[int] = []
    ys: list[float] = []
    for row in source_rows(source_name):
        x_value = int(row.get(x_key, 0))
        y_value = float(row.get(y_key, math.nan))
        if x_key == "tokens_seen" and x_value <= 0:
            continue
        if not math.isfinite(y_value):
            continue
        xs.append(x_value)
        ys.append(y_value)
    return xs, ys


def save_figure(fig, file_name: str) -> Path:
    path = PLOT_DIR / file_name
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    return path


def plot_loss_lines(ax, x_key: str, *, include_val: bool = True) -> None:
    for source_name in SOURCE_ORDER:
        xs, ys = xy_series(source_name, x_key, "train_loss")
        if xs:
            ax.plot(
                xs,
                ys,
                marker="o",
                linewidth=2,
                label=f"{source_name} train",
                color=SOURCE_COLORS.get(source_name),
            )
        if include_val:
            xs, ys = xy_series(source_name, x_key, "val_loss")
            if xs:
                ax.plot(
                    xs,
                    ys,
                    marker="o",
                    linewidth=2,
                    linestyle="--",
                    label=f"{source_name} val",
                    color=SOURCE_COLORS.get(source_name),
                    alpha=0.75,
                )
    ax.grid(True, alpha=0.25)
    ax.set_ylabel("Loss")
    ax.legend()


def loss_to_perplexity(loss: float) -> float:
    if not math.isfinite(loss) or loss > 50:
        return math.nan
    return math.exp(loss)

## 5. Loss By Optimizer Step

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_loss_lines(ax, "global_step")
ax.set_title("Protein Pretraining Loss by Optimizer Step")
ax.set_xlabel("Optimizer step")
loss_by_step_path = save_figure(fig, "loss_by_step.png")
loss_by_step_path

## 6. Loss By Tokens Seen

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_loss_lines(ax, "tokens_seen")
ax.set_title("Protein Pretraining Loss by Tokens Seen")
ax.set_xlabel("Tokens seen")
loss_by_tokens_path = save_figure(fig, "loss_by_tokens.png")
loss_by_tokens_path

## 7. Perplexity By Optimizer Step

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plotted_any = False
for source_name in SOURCE_ORDER:
    xs, losses = xy_series(source_name, "global_step", "train_loss")
    ys = [loss_to_perplexity(loss) for loss in losses]
    pairs = [(x, y) for x, y in zip(xs, ys) if math.isfinite(y)]
    if pairs:
        plotted_any = True
        ax.plot(
            [x for x, _ in pairs],
            [y for _, y in pairs],
            marker="o",
            linewidth=2,
            label=f"{source_name} train",
            color=SOURCE_COLORS.get(source_name),
        )

    xs, losses = xy_series(source_name, "global_step", "val_loss")
    ys = [loss_to_perplexity(loss) for loss in losses]
    pairs = [(x, y) for x, y in zip(xs, ys) if math.isfinite(y)]
    if pairs:
        plotted_any = True
        ax.plot(
            [x for x, _ in pairs],
            [y for _, y in pairs],
            marker="o",
            linewidth=2,
            linestyle="--",
            label=f"{source_name} val",
            color=SOURCE_COLORS.get(source_name),
            alpha=0.75,
        )

if plotted_any:
    ax.set_yscale("log")
ax.set_title("Protein Pretraining Perplexity by Optimizer Step")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Perplexity")
ax.grid(True, alpha=0.25)
ax.legend()
perplexity_by_step_path = save_figure(fig, "perplexity_by_step.png")
perplexity_by_step_path

## 8. Tokens Seen By Optimizer Step

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for source_name in SOURCE_ORDER:
    xs, ys = xy_series(source_name, "global_step", "train_loss")
    token_lookup = {
        int(row["global_step"]): int(row["tokens_seen"])
        for row in source_rows(source_name)
        if int(row.get("tokens_seen", 0)) > 0
    }
    token_pairs = [(step, token_lookup.get(step, 0)) for step in xs]
    token_pairs = [(step, tokens) for step, tokens in token_pairs if tokens > 0]
    if token_pairs:
        ax.plot(
            [step for step, _ in token_pairs],
            [tokens for _, tokens in token_pairs],
            marker="o",
            linewidth=2,
            label=source_name,
            color=SOURCE_COLORS.get(source_name),
        )

ax.set_title("Tokens Seen by Optimizer Step")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Tokens seen")
ax.grid(True, alpha=0.25)
ax.legend()
tokens_by_step_path = save_figure(fig, "tokens_by_step.png")
tokens_by_step_path

## 9. Combined Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

plot_loss_lines(axes[0, 0], "global_step")
axes[0, 0].set_title("Loss by step")
axes[0, 0].set_xlabel("Optimizer step")

plot_loss_lines(axes[0, 1], "tokens_seen")
axes[0, 1].set_title("Loss by tokens")
axes[0, 1].set_xlabel("Tokens seen")

for source_name in SOURCE_ORDER:
    xs, losses = xy_series(source_name, "global_step", "train_loss")
    ys = [loss_to_perplexity(loss) for loss in losses]
    pairs = [(x, y) for x, y in zip(xs, ys) if math.isfinite(y)]
    if pairs:
        axes[1, 0].plot(
            [x for x, _ in pairs],
            [y for _, y in pairs],
            marker="o",
            linewidth=2,
            label=f"{source_name} train",
            color=SOURCE_COLORS.get(source_name),
        )
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Train perplexity by step")
axes[1, 0].set_xlabel("Optimizer step")
axes[1, 0].set_ylabel("Perplexity")
axes[1, 0].grid(True, alpha=0.25)
axes[1, 0].legend()

for source_name in SOURCE_ORDER:
    step_tokens = [
        (int(row["global_step"]), int(row["tokens_seen"]))
        for row in source_rows(source_name)
        if int(row.get("tokens_seen", 0)) > 0
    ]
    if step_tokens:
        axes[1, 1].plot(
            [step for step, _ in step_tokens],
            [tokens for _, tokens in step_tokens],
            marker="o",
            linewidth=2,
            label=source_name,
            color=SOURCE_COLORS.get(source_name),
        )
axes[1, 1].set_title("Tokens seen by step")
axes[1, 1].set_xlabel("Optimizer step")
axes[1, 1].set_ylabel("Tokens seen")
axes[1, 1].grid(True, alpha=0.25)
axes[1, 1].legend()

dashboard_path = save_figure(fig, "metrics_dashboard.png")
dashboard_path

## 10. Output Files

In [ ]:
output_files = [
    loss_by_step_path,
    loss_by_tokens_path,
    perplexity_by_step_path,
    tokens_by_step_path,
    dashboard_path,
]
[str(path) for path in output_files]